# Notebook 00: Environment Setup & Hardware Verification

**Objective**: Initialize and verify the OPPORTUNITY HAR research environment for Google Colab or local execution.

### What this notebook does:
1. **Colab Detection & Repository Bootstrap**: Clones `https://github.com/DhruvikaRajput/Opportunity_HAR.git` to `/content/Opportunity_HAR` if opened in Google Colab.
2. **Import Path Configuration**: Anchors `sys.path` and changes working directory to the repository root so `import src.api` works immediately.
3. **Dependency Installation**: Automatically installs requirements from `requirements.txt`.
4. **Google Drive Mount**: Enables optional mounting of Google Drive (`/content/drive`) to locate dataset files and persist checkpoints.
5. **Dataset Location & Verification**: Automatically searches local and Google Drive locations for the 24 official `.dat` recording files.
6. **Hardware Check**: Reports active compute device (CPU or CUDA GPU) and memory.

## 1. Repository Setup & Import Path Configuration
Run this cell first. It automatically configures `sys.path` and installs dependencies.

In [ ]:
# ==============================================================================
# 1. ENVIRONMENT DETECTION & REPOSITORY BOOTSTRAP
# ==============================================================================
import sys
import os
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    REPO_DIR = Path("/content/Opportunity_HAR")
    if not REPO_DIR.exists():
        print("[INFO] Cloning Opportunity_HAR repository from GitHub...")
        !git clone https://github.com/DhruvikaRajput/Opportunity_HAR.git {REPO_DIR}
    else:
        print(f"[INFO] Repository already present at {REPO_DIR}. Pulling latest changes...")
        !git -C {REPO_DIR} pull origin main

    # Set repository root at front of sys.path
    if str(REPO_DIR) not in sys.path:
        sys.path.insert(0, str(REPO_DIR))

    os.chdir(REPO_DIR)
    !pip install -q -r requirements.txt
else:
    current = Path.cwd()
    REPO_DIR = current.parent if current.name == "notebooks" else current
    if str(REPO_DIR) not in sys.path:
        sys.path.insert(0, str(REPO_DIR))
    os.chdir(REPO_DIR)

print(f"Active Working Directory : {Path.cwd()}")

# Ensure fresh import of src.api
import importlib
if "src.api" in sys.modules:
    import src.api
    importlib.reload(src.api)
else:
    import src.api as api

import src.api as api
status = api.setup_project()
print(f"High-Level API Loaded    : src.api (locate_dataset available: {hasattr(api, 'locate_dataset')})")
print(f"Compute Device Detected  : {status['device_name']} (CUDA Available: {status['cuda_available']})")

## 2. Dataset Discovery & Verification
The 24 raw .dat recordings (850+ MB) are intentionally kept outside GitHub.
This cell searches data/raw/ and common Google Drive locations (auto-mounting Drive if needed).

In [ ]:
ds_status = api.locate_dataset()

print("=" * 65)
print("DATASET VERIFICATION STATUS")
print("=" * 65)
print(f"Found Complete Set : {ds_status['found']}")
print(f"Recordings Found   : {ds_status['files_found']}/{ds_status['total_expected']}")
print(f"Directory Path     : {ds_status['dataset_dir']}")
print("-" * 65)
print(ds_status["message"])
print("=" * 65)

if not ds_status["found"] and IN_COLAB:
    print("\n[ACTION REQUIRED FOR COLAB]")
    print("1. Upload the 24 OPPORTUNITY .dat files to your Google Drive under:")
    print("   'My Drive/Opportunity_HAR/data/raw/' or 'My Drive/Opportunity/'")
    print("2. Make sure you ran the Google Drive mount cell above.")
    print("3. Re-run this cell to verify.")

## 4. Hardware & GPU Acceleration Details

In [ ]:
import torch

print("=" * 60)
print("COMPUTE HARDWARE AUDIT")
print("=" * 60)
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"Device Type   : CUDA GPU ({props.name})")
    print(f"Total VRAM    : {props.total_memory / (1024**3):.2f} GB")
    print(f"Compute Arch  : sm_{props.major}{props.minor}")
    print("Status        : READY FOR GPU NEURAL NETWORK TRAINING")
else:
    print("Device Type   : CPU")
    print("Notice        : Full neural training (CNN, GNN) requires a GPU.")
    print("To enable GPU in Colab: Runtime -> Change runtime type -> T4 GPU.")
print("=" * 60)